# Tool + 기본 Agent

In [45]:
from pprint import pprint

from dotenv import load_dotenv
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
MODEL_NAME="gemini-3.6-flash"

### Tool 개념

- Tool은 LLM이 외부 세계와 상호작용할 수 있게 해주는 함수이다
- LLM 자체는 텍스트만 생성할 수 있지만, Tool을 통해 웹 검색, 계산, 외부 API 조회 등을 수행할 수 있다
- LLM이 "이 Tool을 호출해야겠다"고 판단하면, Tool 이름과 인자를 반환한다

```
사용자 질문 → LLM이 판단 → Tool 호출 필요?
  → Yes: Tool 이름 + 인자 반환 → Tool 실행 → 결과를 LLM에 전달 → 최종 응답
  → No: 바로 텍스트 응답
```

예를 들어 사용자가 "서울 날씨 알려줘"라고 하면, LLM은 스스로 날씨를 알 수 없다. 대신 "날씨 검색 Tool을 '서울'이라는 인자로 호출해야겠다"고 판단한다. 개발자가 실제로 Tool을 실행하고 결과를 LLM에 돌려주면, LLM이 그 결과를 자연어로 정리하여 답변한다.

---


### @tool 데코레이터

- Python 함수를 LangChain Tool로 변환하는 가장 간단한 방법
- 함수의 docstring이 Tool의 설명(description)이 된다
- `@tool` 데코레이터를 붙이면 일반 함수가 LangChain Tool 객체로 변환된다. LLM은 이 Tool의 `name`, `description`, `args_schema`를 보고 어떤 Tool을 어떤 인자로 호출할지 판단한다. 따라서 **docstring(설명)과 타입 힌트가 매우 중요하다**.

`@tool(parse_docstring=True)`를 사용하고 docstring을 Google 스타일의 `Args:` 형식으로 작성하면, LangChain이 각 parameter의 설명을 자동으로 추출해 `args_schema`의 `description`에 넣어준다. 별도의 Pydantic 스키마 없이도 인자의 의미를 LLM에게 전달할 수 있다.

```python
@tool(parse_docstring=True)
def search_weather(city: str) -> str:
    """도시의 현재 날씨를 검색한다.

    Args:
        city: 날씨를 검색할 도시 이름
    """
```

`search_weather.args_schema.model_json_schema()`로 자동 생성된 parameter description을 확인할 수 있다.


#### Tool 설계 원칙

| 원칙 | 설명 |
|------|------|
| 명확한 이름 | `search_weather` > `func1` |
| 구체적인 설명 | "주어진 도시의 현재 날씨 정보를 검색한다" > "데이터를 가져온다" |
| 타입 힌트 필수 | `city: str` — LLM이 어떤 값을 넣어야 하는지 알 수 있다 |
| 예시 포함 | docstring에 입력 예시를 넣으면 정확도가 높아진다 |
| 에러 메시지 | Tool 실행 실패 시 LLM이 이해할 수 있는 메시지를 반환한다 |

**Tool을 나누는 기준**: API 엔드포인트가 아니라 **LLM이 docstring만 보고 언제 쓸지 판단할 수 있는 단위**로 나눈다. 용도가 다르면 분리하고(검색 vs 상세 조회), 파라미터 하나 차이면 합쳐도 된다. 너무 많으면 선택 정확도가 떨어지고, 너무 합치면 파라미터가 복잡해져서 LLM이 헷갈린다.

In [12]:
@tool(parse_docstring=True)
def search_weather(city: str) -> str:
    """주어진 도시의 현재 날씨를 검색한다.

    Args:
        city: 날씨를 검색할 도시 이름. 예: '서울', '부산'
    """
    weather_data = {
        "서울": "맑음, 22도, 습도 45%",
        "부산": "흐림, 19도, 습도 72%",
        "제주": "비, 17도, 습도 88%",
    }
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

# # Tool 정보 확인
print(f"이름: {search_weather.name}")
print(f"설명: {search_weather.description}")
pprint(f"스키마: {search_weather.args_schema.model_json_schema()}")

# Tool 직접 호출
print(f"\n서울 날씨: {search_weather.invoke({'city': '서울'})}")

# @Tool이 없을 때 호출은 ()로 하지만, @Tool이 되면 invoke를 사용한다.
# print(f"\n서울 날씨: {search_weather(city="서울")}")

이름: search_weather
설명: 주어진 도시의 현재 날씨를 검색한다.
("스키마: {'description': '주어진 도시의 현재 날씨를 검색한다.', 'properties': {'city': "
 '{\'description\': "날씨를 검색할 도시 이름. 예: \'서울\', \'부산\'", \'title\': \'City\', '
 "'type': 'string'}}, 'required': ['city'], 'title': 'search_weather', 'type': "
 "'object'}")

서울 날씨: 맑음, 22도, 습도 45%


---


### Tool 바인딩

- LLM에 사용할 수 있는 Tool 목록을 알려주는 것
- `bind_tools()`를 사용한다

`bind_tools()`로 Tool을 바인딩하면, LLM은 사용자의 질문을 보고 두 가지 중 하나를 선택한다.

1. **Tool 호출이 필요한 경우** — `tool_calls`에 호출할 Tool 정보를 담아서 반환 (content는 비어 있을 수 있음)
2. **Tool 호출이 불필요한 경우** — 일반 텍스트 응답을 content에 담아서 반환

중요한 점은, `bind_tools()`만으로는 **Tool이 실제로 실행되지 않는다**. LLM은 "이 Tool을 이 인자로 호출해줘"라고 요청할 뿐이고, 실제 실행은 개발자가 해야 한다.

In [13]:
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
llm_with_tools = llm.bind_tools([search_weather])

# Tool 호출이 필요한 경우
response = llm_with_tools.invoke("서울 날씨 알려줘")
print("content:", response.content)        # 비어 있을 수 있음
print("tool_calls:", response.tool_calls)   # Tool 호출 정보

print()

# Tool 호출이 불필요한 경우
response2 = llm_with_tools.invoke("안녕하세요")
print("content:", response2.content)        # 일반 응답
print("tool_calls:", response2.tool_calls)  # 빈 리스트

content: []
tool_calls: [{'name': 'search_weather', 'args': {'city': '서울'}, 'id': 'call_259367', 'type': 'tool_call'}]

content: [{'type': 'text', 'text': '안녕하세요! 무엇을 도와드릴까요?', 'extras': {'signature': 'EtUCCtICARFNMg9JcD6gF5pmswO65vft6e/CBXHY8KEFy3ZNIUyhyIIRMhKWragzRafq4K0h694v9OL/vkxelyJHDYPJ64xg1m7ll/pTG6oAXOi7CLg3WbcuVXzCa+FYLMLSOyoptYNBOyaen3SMUWhloApun/n7iFU0DdHrBLNcumaYaUuceUtiSwPFiS4YfSuFd3xs0t3M5KG6pI9nKS1gSzDuC4FGl1Jingld0EuiwSUyDSJCLQBf4TcXUsZ616uFIE6KClbFIKgXlKCBW86fZguQC0Ja8QcQvOxDL6+Sa05FZHt1x4CIJSFWhy7LYk84ZbXgxUhyEQh0Pw91My1TX3VzxX41bXgfvrfv1Go1mgV5lNz1tv/W/IB6ENPJustS2Er15fJDsRL+1J/4BaHPWPqvawMr5zFDWHdnVfhs64afN20e6BgCS1mkjlyjmlcftzO3MhM='}}]
tool_calls: []


---


### Function Calling 비교: Google Gen AI SDK vs LangChain

| 비교 항목 | Google Gen AI SDK | LangChain @tool |
|-----------|-------------|------------------|
| Tool 정의 | JSON 스키마 직접 작성 | Python 함수 + docstring |
| 파라미터 | 수동으로 properties 정의 | 타입 힌트에서 자동 추출 |
| 결과 확인 | provider SDK 고유 응답 구조 | `AIMessage.tool_calls`로 표준화 |
| 모델 교체 | API별 코드 재작성 | `ChatGoogleGenerativeAI` → 다른 ChatModel 한 줄 변경 |

---


### Tool 호출 루프

Agent는 모델이 Tool을 요청하는 동안 **모델 호출 → Tool 실행 → `ToolMessage` 추가 → 모델 재호출** 과정을 반복한다. 모델은 Tool을 직접 실행하지 않으므로 애플리케이션이 `tool_calls`를 읽고 해당 Tool을 실행해야 한다.

```
사용자 입력 → 모델 판단 → Tool 실행 → 결과 전달 → 모델 재호출 → 최종 응답
```

In [18]:
tools = [search_weather]
tool_map = {tool.name: tool for tool in tools}
llm_with_tools = llm.bind_tools(tools)

MAX_ITERATIONS = 5


def run_agent(user_input: str) -> str:
    messages = [HumanMessage(content=user_input)]

    for _ in range(MAX_ITERATIONS):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tool_call in response.tool_calls:
            # toolmap에서 함수의 이름을 key값으로 가지는 value인 함수를 가지고온다.
            tool_result = tool_map[tool_call["name"]].invoke(tool_call["args"])
            print(f'{tool_call["name"]}실행 - {tool_call["args"]} - {tool_result}')
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"],
                )
            )

    return "최대 반복 횟수를 초과했습니다."

In [19]:
# 다양한 질문으로 테스트
questions = [
    "서울 날씨 어때?",
    "서울이랑 부산 날씨 비교해줘",
    "안녕하세요!",
]

for q in questions:
    print(f"\n사용자: {q}")
    answer = run_agent(q)
    print(f"Agent: {answer}")


사용자: 서울 날씨 어때?
search_weather실행 - {'city': '서울'} - 맑음, 22도, 습도 45%
Agent: [{'type': 'text', 'text': '현재 서울의 날씨는 **맑음**이며, 기온은 **22℃**, 습도는 **45%**입니다.', 'extras': {'signature': 'EvsBCvgBARFNMg/ttZ5zlayBl5k4RkOZKELXfoo8W52znWplQGA5LoG39BHhd+3dMunsoCPAzUcV0lUHMEqRDQxCraofZg2+HTPD6BGwb6Od+F87q9a9A3vGZ7pbjrooMDVZHNq5Vv+EFsj5PAogw3G1sqerFvpEDhAhUk7u110lf2c6nBFDSsBMYtzzI7aoiYJKgLbjNqlgU4x+/XkgdNen5Wr2Ke5txlhgFHwXw+XflQ70pg/lJxj1/fJZtC1xw/Epe3tYy/44Sm9YFVfd6FytEXfPOq9OPZUoNTWAzhEfgxQwX19nnOQHBuO9xLo64Erh6BOAmRzZ/FbHthI='}}]

사용자: 서울이랑 부산 날씨 비교해줘
search_weather실행 - {'city': '서울'} - 맑음, 22도, 습도 45%
search_weather실행 - {'city': '부산'} - 흐림, 19도, 습도 72%
Agent: [{'type': 'text', 'text': '현재 **서울**과 **부산**의 날씨 비교 결과입니다.\n\n* **서울**: 맑음 / **22°C** / 습도 **45%**\n* **부산**: 흐림 / **19°C** / 습도 **72%**\n\n서울은 부산보다 기온이 3°C 높아 더 따뜻하고 맑은 날씨를 보이는 반면, 부산은 흐리고 습도가 높아 다소 쌀쌀하게 느껴질 수 있습니다.', 'extras': {'signature': 'EoMCCoACARFNMg/Td9rN6Z/dpkzvy+ll5IINB7g+gsRB64ERS1JcqDVrTgBeXXBciADbPHxbbRzClmtMgIuCgLwtWt/CmWDld32

> **참고**: 모델은 한 번에 여러 Tool을 요청하거나 Tool 결과를 보고 추가 Tool을 요청할 수 있다. 반복 횟수를 제한해 무한 호출을 방지해야 한다. 이 반복 구조는 이후 LangGraph에서 `ToolNode`와 `tools_condition`으로 구성한다.

---

### 실습 문제

#### 영화 목록과 상세 조회 Agent

TMDB API와 LangChain Tool을 사용해 영화 목록을 조회하고, 후속 질문에서 특정 영화의 상세 정보를 조회하는 Agent를 구현하세요.

```.env
TMDB_API_KEY=...
```

다음 Tool을 구현합니다.

| Tool | 역할 |
|---|---|
| `get_movie_list` | 인기, 현재 상영, 개봉 예정 목록에서 영화 ID, 제목, 개봉일을 조회한다 |
| `get_movie_detail` | 영화 ID로 줄거리, 장르, 러닝타임 등 상세 정보를 조회한다 |

목록 Tool은 상세 정보를 포함하지 않습니다. 후속 질문에 답하려면 목록 결과의 영화 ID를 사용해 `get_movie_detail`을 호출해야 합니다.

다음 두 질문을 **같은 세션**에서 순서대로 실행합니다.

```text
현재 상영 중인 영화 5개를 알려줘.
그중 첫 번째 영화의 줄거리와 러닝타임을 알려줘.
```

다른 세션에서 바로 `"그중 첫 번째 영화의 줄거리를 알려줘."`라고 질문했을 때 이전 목록을 알 수 없다고 답하는지도 확인하세요.

In [ ]:
# 내가 지금 뭘 하고싶지?
# 1. 상영중인 영화 5개 가져오기.
# get_movies() 함수를 실행한다.

# 2. 첫번째 영화의 줄거리와 러닝타임 알려줘.
# get_movie_by_id() 함수를 실행한다. <- id로 첫번째 get_movies()에서 가져온 movie의 id를 넣겠다.

# 결과가 나온다.



In [33]:
import requests
import os
from pprint import pprint

load_dotenv()

TMDB_TOKEN = os.getenv('TMDB_TOKEN')

def get_movies(size: int = 10):
    """
        TMDB API를 활용해서 상영중인 영화의 목록을 가져오는 함수.
    """

    URL = "https://api.themoviedb.org/3/movie/now_playing"
    headers = {
        'authorization' : f'Bearer {TMDB_TOKEN}'
    }
    try:
        response = requests.get(URL, headers=headers)
        response.raise_for_status
        # print(response.json())
        data = response.json()
        result = [
            {
                'id' : movie.get('id'),
                'title' : movie.get('title'),
                'release_date' : movie.get('release_date')
            }
            for movie in data.get('results')
        ]
        return result[:size]
    
    except Exception as e:
        print(e)
        return {}



def get_movie_by_id(id: int):
    """
        TMDB API를 활용해서 주어진 ID에 해당하는 영화의 상세 정보를 가져오는 함수.
    """

    URL = f"https://api.themoviedb.org/3/movie/{id}"
    headers = {
        'authorization' : f'Bearer {TMDB_TOKEN}'
    }
    try:
        response = requests.get(URL, headers=headers)
        response.raise_for_status
        # print(response.json())
        data = response.json()
        return data

    except Exception as e:
        print(e)
        return {}


In [35]:
# 실행 과정
# 내가 지금 뭘 하고싶지?
# 1. 상영중인 영화 5개 가져오기.
# get_movies() 함수를 실행한다.
data = get_movies(size=5)

# 2. 첫번째 영화의 줄거리와 러닝타임 알려줘.
# get_movie_by_id() 함수를 실행한다. <- id로 첫번째 get_movies()에서 가져온 movie의 id를 넣겠다.
first_id = data[0].get('id')
print(first_id)

movie = get_movie_by_id(first_id)
# pprint(movie)

# 결과가 나온다.
print(movie.get('overview'))





969681
Fighting crime full-time as Spider-Man in a world that doesn't remember him—and the pressure of seeing his old friends move on without him—sparks a change in Peter Parker he may not have the power to control. But that transformation might also be the only thing that can stop a shocking new threat to the city and those he loves - a powerful villain no one can even see.


In [ ]:
# llm한테 맡겨보자.
# 1. llm이 함수를 호출할 수 있도록 만들어야 합니다.
tools = []

In [57]:
import requests
import os
from pprint import pprint

load_dotenv()

TMDB_TOKEN = os.getenv('TMDB_TOKEN')

@tool(parse_docstring=True)
def get_movies_tool(category: str, size: int = 10):
    """TMDB API를 활용해서 상영중인 영화의 목록을 가져오는 함수.

    Args:
        category: 어떤 영화의 목록을 가져올건지 정하는 파라미터. 예 : now_playing - 현재 상영중, popular - 인기있는, top_rated - 순위권인, upcoming - 개봉 예정인.
        size: 영화 목록의 사이즈. 예: 5, 3, 7
    """

    URL = f"https://api.themoviedb.org/3/movie/{category}"
    headers = {
        'authorization' : f'Bearer {TMDB_TOKEN}'
    }
    try:
        response = requests.get(URL, headers=headers)
        response.raise_for_status
        # print(response.json())
        data = response.json()
        result = [
            {
                'id' : movie.get('id'),
                'title' : movie.get('title'),
                'release_date' : movie.get('release_date')
            }
            for movie in data.get('results')
        ]
        return result[:size]
    
    except Exception as e:
        print(e)
        return {}


@tool(parse_docstring=True)
def get_movie_by_id_tool(id: int):
    """TMDB API를 활용해서 주어진 ID에 해당하는 영화의 상세 정보를 가져오는 함수.

    Args:
        id: TMDB API에서 활용되는 영화 id. 예 : 969681
    """

    URL = f"https://api.themoviedb.org/3/movie/{id}"
    headers = {
        'authorization' : f'Bearer {TMDB_TOKEN}'
    }
    try:
        response = requests.get(URL, headers=headers)
        response.raise_for_status
        # print(response.json())
        data = response.json()
        return data

    except Exception as e:
        print(e)
        return {}

print(get_movies_tool.name)
pprint(get_movies_tool.args_schema.model_json_schema())
print(get_movie_by_id_tool.name)
print(get_movie_by_id_tool.args_schema.model_json_schema())

get_movies_tool
{'description': 'TMDB API를 활용해서 상영중인 영화의 목록을 가져오는 함수.',
 'properties': {'category': {'description': '어떤 영화의 목록을 가져올건지 정하는 파라미터. 예 : '
                                            'now_playing - 현재 상영중, popular - '
                                            '인기있는, top_rated - 순위권인, upcoming '
                                            '- 개봉 예정인.',
                             'title': 'Category',
                             'type': 'string'},
                'size': {'default': 10,
                         'description': '영화 목록의 사이즈. 예: 5, 3, 7',
                         'title': 'Size',
                         'type': 'integer'}},
 'required': ['category'],
 'title': 'get_movies_tool',
 'type': 'object'}
get_movie_by_id_tool
{'description': 'TMDB API를 활용해서 주어진 ID에 해당하는 영화의 상세 정보를 가져오는 함수.', 'properties': {'id': {'description': 'TMDB API에서 활용되는 영화 id. 예 : 969681', 'title': 'Id', 'type': 'integer'}}, 'required': ['id'], 'title': 'get_movie_by_id_tool', 'type': 'object'

In [58]:
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
tools = [get_movies_tool, get_movie_by_id_tool]

tool_map = {
    'get_movies_tool' : get_movies_tool,
    'get_movie_by_id_tool' : get_movie_by_id_tool
}

tool_map = {
    func.name : func
    for func in tools
}

llm_with_movie_tool = llm.bind_tools(tools)


# # 실행 과정
# # 내가 지금 뭘 하고싶지?
# # 1. 상영중인 영화 5개 가져오기.
# # get_movies() 함수를 실행한다.
# data = get_movies(size=5)

# # 2. 첫번째 영화의 줄거리와 러닝타임 알려줘.
# # get_movie_by_id() 함수를 실행한다. <- id로 첫번째 get_movies()에서 가져온 movie의 id를 넣겠다.
# first_id = data[0].get('id')
# print(first_id)

# movie = get_movie_by_id(first_id)
# # pprint(movie)

# # 결과가 나온다.
# print(movie.get('overview'))

def run_agent(request: str, max_attempt: int = 5):
    messages = [HumanMessage(content=request)]

    for _ in range(max_attempt):

        response = llm_with_movie_tool.invoke(messages)

        messages.append(response)

        # 만약 일반 응답이면 그냥 실행해줘
        if not response.tool_calls:
            pprint(messages)
            return response.content

        # 만약 함수를 실행하라는 명령이면 함수를 실행해줘.
        # 그리고 해당 응답을 담아서 다시 요청해줘.

        for tool_call in response.tool_calls:
            func_name = tool_call.get('name')
            args = tool_call.get('args')
            tool_id = tool_call.get('id')
            
            func = tool_map.get(func_name)

            result = func.invoke(args)
            messages.append(
                ToolMessage(
                    content = str(result),
                    tool_call_id = tool_id
                )
            )


result = run_agent('상영중인 영화 5개 가져와줘.')
print(result)

result = run_agent('첫번째 영화의 줄거리와 러닝타임 알려줘.')
print(result)

[HumanMessage(content='상영중인 영화 5개 가져와줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_movies_tool', 'arguments': '{"category": "now_playing", "size": 5}'}, '__gemini_function_call_thought_signatures__': {'call_594828': 'EoMDCoADARFNMg/AKGJ200LB1CyXHN9Dqx1+wFdXB3ggd3JwlBOXWkyyTJToRYZge0UTZ/5M0C7GGR/KIAuppCOS144ZNiy1EYHtEIjrlSG24PQUrb6tyy0rmJ0qPlS8OV2/ZefCDKCrwY3lQwdN442ChoIq9jOssRRJe14ankdOh+w35s/4x2jw6XaSQPIWFIKGUYdLsKcSw+WPHlbzW8L8PsuDyyKUQ0Tq2CJS3h/kooyUY5VjlyrCCSnLRGIM3qBDQpbGwSH0xSqHdR4ChNNCqAhajCVA7/c17swPiBnk9VYN5Q7jFw1hrElbNQrMiQNkXy4lwa/P+jvsG/9bVzEH1x2qbnLX7eAHGhf4xbIJj7+8sXM/J9k+m7fSk0VoJd7RPChmPr9aCDeIpO0A+Gru6cRFNQwcORoeaSO8+Xuv5DR7MvLNyunpGKv9abLpFcpoIwT8ZYAHaLA2sE/z/ai5xG3EbLeuTZOMdHqmMVYoqaLCJxi74IQKbpfEBZqHEY+RXhRi'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a060ab-51e1-7fc0-89ea-b3781d9ac4

In [65]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# 대화 내역을 기억하는 세션을 만들어서 대화가 지속적으로 이어질 수 있도록 하고 싶다.

llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
tools = [get_movies_tool, get_movie_by_id_tool]

tool_map = {
    'get_movies_tool' : get_movies_tool,
    'get_movie_by_id_tool' : get_movie_by_id_tool
}

tool_map = {
    func.name : func
    for func in tools
}

llm_with_movie_tool = llm.bind_tools(tools)


momory = {
    'user-1' : InMemoryChatMessageHistory(),
    'user-2' : InMemoryChatMessageHistory()
}

def run_agent(user_id: str, request: str, max_attempt: int = 5):
    user_memory = momory.get(user_id)
    user_message = HumanMessage(content=request)
    
    user_memory.add_message(user_message)

    # messages = [
    #     *user_memory.messages, 
    #     ]

    for _ in range(max_attempt):

        response = llm_with_movie_tool.invoke(user_memory.messages)

        # messages.append(response)
        user_memory.add_message(response)

        # 만약 일반 응답이면 그냥 실행해줘
        if not response.tool_calls:
            # pprint(messages)
            pprint(user_memory.messages)
            return response.content

        # 만약 함수를 실행하라는 명령이면 함수를 실행해줘.
        # 그리고 해당 응답을 담아서 다시 요청해줘.

        for tool_call in response.tool_calls:
            func_name = tool_call.get('name')
            args = tool_call.get('args')
            tool_id = tool_call.get('id')
            
            func = tool_map.get(func_name)

            result = func.invoke(args)
            tool_message = ToolMessage(
                content = str(result),
                tool_call_id = tool_id
                )
            # messages.append(tool_message)
            user_memory.add_message(tool_message)


result = run_agent('user-1', '상영중인 영화 5개 가져와줘.')
print(result)
print()
result = run_agent('user-2', '첫번째 영화의 줄거리와 러닝타임 알려줘.')
print(result)

[HumanMessage(content='상영중인 영화 5개 가져와줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_movies_tool', 'arguments': '{"size": 5, "category": "now_playing"}'}, '__gemini_function_call_thought_signatures__': {'call_306892': 'EpsECpgEARFNMg9oN7njP7SEk1uq6oK51HI8731ibaGkg37XgDpNH22NuZopS4TXwT+ovmqZqRrHTg1srea4XyKtu46ibQ3JEe+k90lkyrpxR1zZZzJ64jqeQM7NIdECtMkwmFMBRRLfjoFy77aAHvVlIl4I1D8Ove9RSKD+jTaMlBPy2FZrcjBij9zUTfBsdGMCnB83JjcJeSG3AEUyks8XC0lhdVd1suM5ednV3XS/NK8LA6eahi12nv8Dsn/OfghYK9fGq1upfDZdVIOLpUm1ta72ZcM13As5BX4nKGbWGCXfEXSAoBLmaBF/7IqGvcp/q+x6kK8ICYoYLTg/EOK/KyUda94jIPgzHqw0nQsAcp2jO8OaPuymAvXSVIYbQLMDPh/K4w1n1Lev+tjdiE7lg1wOAIroeLYCuG/YymOS4gFACLObk0aTRAZeKVoQI0uHfGFCBR7sdNkhlL3MBVKLg0oGvSL5GDVFUHNBOE726/y1OS9luAgCWXxxrE8wYkVsvCQxsay/UGRsBURbD7fzcVUbmS41ZFNCYR3h1t54MEcBTBVCD7B7oG70QqyyiN/gC38CpXWbVwV78Px+5V+8khIbfMbmZhwj2JBXTR8vAOk+2kN8EwoKxjizZTK/dX4ZzLIDw2w4jcX7ALObk0DRYCy9MwfA6Nx0L43X94/DjLkaamCivX7eHz4ERUwT20